# Fraud Detection on a Transaction Graph: XGBoost vs. Graph Neural Networks

**Dataset:** Elliptic Bitcoin dataset: 203,769 transactions (nodes), 234,355 payment flows (edges), and 165 features per transaction.
About 2% are labeled illicit, 21% licit, and the rest unknown, so the data is highly imbalanced and partly labeled.

**Questions**
1. Does graph structure help beyond per-transaction features? Compare **XGBoost** (tabular), **GCN / GraphSAGE** (graph), and **hybrids** (XGBoost + neighbor features, XGBoost + GNN embeddings).
2. How do models trade off **catching fraud vs. blocking legitimate users**? Use PR-AUC, F1 at a validation-tuned threshold, and **recall at ≥90% precision**.
3. Do models hold up **over time**? Train on the past, test on the future; the dark-market shutdown around time step 43 is a natural distribution shift.

**Temporal split** (no leakage from the future): train steps 1–29, validation 30–34 (early stopping and thresholds), test 35–49.

**Runtime:** about 10 minutes on a Colab T4 GPU; also runs on CPU.

Dataset: Weber et al., *Anti-Money Laundering in Bitcoin: Experimenting with Graph Convolutional Networks for Financial Forensics* (KDD Workshop 2019), loaded via PyTorch Geometric.

In [ ]:
!pip -q install torch_geometric xgboost

In [ ]:
import os, json, time, random
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
print("device:", dev)

## 1. Load the graph and build the temporal split

In [ ]:
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.utils import to_undirected, scatter

ds = EllipticBitcoinDataset(root="data/elliptic")
data = ds[0]
feat_path = [p for p in ds.raw_paths if "features" in p][0]
ts = pd.read_csv(feat_path, header=None, usecols=[1])[1].values          # time step per node (same order as data.x)
y_all = data.y.numpy()                                                    # 0 = licit, 1 = illicit, 2 = unknown
lab = y_all != 2

assert (data.train_mask.numpy() == (lab & (ts < 35))).all(), "time-step alignment check failed"
tr_m, va_m, te_m = lab & (ts < 30), lab & (ts >= 30) & (ts < 35), lab & (ts >= 35)
ytr, yv, yt = y_all[tr_m], y_all[va_m], y_all[te_m]
N = data.num_nodes
print(f"nodes {N:,} | edges {data.edge_index.shape[1]:,} | features {data.x.shape[1]}")
for n, m in [("train", tr_m), ("val", va_m), ("test", te_m)]:
    print(f"{n:5s}: {m.sum():6,} labeled | illicit {y_all[m].mean():.1%}")

In [ ]:
X_raw = data.x.numpy()
scaler = StandardScaler().fit(X_raw[tr_m])
X_all = scaler.transform(X_raw).astype(np.float32)
X_loc = X_all[:, :93]                       # 93 local (per-transaction) features; the rest are 1-hop aggregates

# Hand-built graph features: mean and max of neighbours' local features (in + out edges)
ei = to_undirected(data.edge_index, num_nodes=N)
xl = torch.tensor(X_loc)
nb_mean = scatter(xl[ei[0]], ei[1], dim=0, dim_size=N, reduce="mean").numpy()
nb_max = scatter(xl[ei[0]], ei[1], dim=0, dim_size=N, reduce="max").numpy()
deg = np.bincount(ei[1].numpy(), minlength=N)[:, None].astype(np.float32)
X_nb = np.hstack([X_all, nb_mean, nb_max, np.log1p(deg)])
print(X_loc.shape, X_all.shape, X_nb.shape)

## 2. Evaluation helpers
Thresholds are chosen on **validation** and then applied to test: one that maximizes F1, and one for precision ≥ 90% (few legitimate users wrongly blocked).

In [ ]:
RES, PREDS = {}, {}

def pick_thresholds(y, p, target_precision=0.90):
    pr, rc, th = precision_recall_curve(y, p)
    pr, rc = pr[:-1], rc[:-1]
    t_f1 = th[np.argmax(2 * pr * rc / (pr + rc + 1e-12))]
    ok = np.where(pr >= target_precision)[0]
    t_p90 = th[ok[np.argmax(rc[ok])]] if len(ok) else 1.0
    return float(t_f1), float(t_p90)

def evaluate(name, p_val, p_test):
    t_f1, t_p90 = pick_thresholds(yv, p_val)
    yhat, yhat90 = (p_test >= t_f1).astype(int), (p_test >= t_p90).astype(int)
    RES[name] = dict(
        pr_auc=average_precision_score(yt, p_test), f1=f1_score(yt, yhat),
        precision=precision_score(yt, yhat, zero_division=0), recall=recall_score(yt, yhat),
        precision_at_p90_thr=precision_score(yt, yhat90, zero_division=0), recall_at_p90_thr=recall_score(yt, yhat90))
    PREDS[name] = (p_test, t_f1)
    print(f"{name:28s} " + " | ".join(f"{k} {v:.3f}" for k, v in RES[name].items()))

## 3. Tabular baselines

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(X_all[tr_m], ytr)
evaluate("LogReg (all features)", lr.predict_proba(X_all[va_m])[:, 1], lr.predict_proba(X_all[te_m])[:, 1])

spw = (ytr == 0).sum() / (ytr == 1).sum()
def xgb(X, name):
    m = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                      scale_pos_weight=spw, eval_metric="aucpr", early_stopping_rounds=100, tree_method="hist",
                      device="cuda" if dev == "cuda" else "cpu", random_state=SEED)
    m.fit(X[tr_m], ytr, eval_set=[(X[va_m], yv)], verbose=False)
    evaluate(name, m.predict_proba(X[va_m])[:, 1], m.predict_proba(X[te_m])[:, 1])
    return m

xgb(X_loc, "XGBoost (local features)")
xgb_all = xgb(X_all, "XGBoost (all features)")

## 4. Graph neural networks (GCN, GraphSAGE)
Full-batch, class-weighted loss, early stopping on validation PR-AUC. Transductive setting: the whole graph is visible, and only train labels are used.

In [ ]:
from torch_geometric.nn import GCNConv, SAGEConv

class GNN(nn.Module):
    def __init__(self, conv, d_in, h=128, p=0.3):
        super().__init__(); self.c1, self.c2, self.out, self.p = conv(d_in, h), conv(h, h), nn.Linear(h, 2), p
    def embed(self, x, e):
        x = F.dropout(F.relu(self.c1(x, e)), self.p, self.training)
        return F.relu(self.c2(x, e))
    def forward(self, x, e):
        return self.out(F.dropout(self.embed(x, e), self.p, self.training))

x_t, e_t, y_t = torch.tensor(X_all).to(dev), ei.to(dev), torch.tensor(y_all).to(dev)
tr_i, va_i = torch.tensor(np.where(tr_m)[0]).to(dev), np.where(va_m)[0]

def train_gnn(conv, name, epochs=400, patience=50):
    torch.manual_seed(SEED)
    m = GNN(conv, X_all.shape[1]).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=0.005, weight_decay=5e-4)
    w = torch.tensor([1.0, float(spw)], device=dev)
    best, bad, state, t0 = -1, 0, None, time.time()
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        F.cross_entropy(m(x_t, e_t)[tr_i], y_t[tr_i], weight=w).backward(); opt.step()
        m.eval()
        with torch.no_grad(): pv = torch.softmax(m(x_t, e_t), 1)[:, 1].cpu().numpy()[va_i]
        ap = average_precision_score(yv, pv)
        if ap > best: best, bad, state = ap, 0, {k: v.detach().clone() for k, v in m.state_dict().items()}
        else: bad += 1
        if bad >= patience: break
    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        p = torch.softmax(m(x_t, e_t), 1)[:, 1].cpu().numpy(); emb = m.embed(x_t, e_t).cpu().numpy()
    print(f"  {name}: stopped at epoch {ep}, best val PR-AUC {best:.3f}, {time.time() - t0:.0f}s")
    evaluate(name, p[va_m], p[te_m])
    return emb

train_gnn(GCNConv, "GCN")
sage_emb = train_gnn(SAGEConv, "GraphSAGE")

## 5. Hybrids: tabular model + graph information

In [ ]:
xgb(X_nb, "XGBoost + neighbor features")
xgb(np.hstack([X_all, sage_emb]), "XGBoost + GraphSAGE emb.")

## 6. Results, PR curves, and robustness over time

In [ ]:
import matplotlib.pyplot as plt
tab = pd.DataFrame(RES).T.sort_values("pr_auc", ascending=False).round(3)
display(tab)
best_name = tab.index[0]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for n, (p, _) in PREDS.items():
    pr, rc, _ = precision_recall_curve(yt, p)
    ax[0].plot(rc, pr, label=f"{n} (AP {RES[n]['pr_auc']:.2f})", lw=2 if n == best_name else 1)
ax[0].set_xlabel("Recall (illicit caught)"); ax[0].set_ylabel("Precision"); ax[0].set_title("Test PR curves (steps 35–49)")
ax[0].legend(fontsize=8)

steps = np.arange(35, 50); tt = ts[te_m]
for n in [best_name, "GraphSAGE", "XGBoost (local features)"]:
    if n not in PREDS: continue
    p, thr = PREDS[n]
    f1s = [f1_score(yt[tt == s], (p[tt == s] >= thr).astype(int), zero_division=0) for s in steps]
    ax[1].plot(steps, f1s, marker="o", label=n)
ax[1].axvline(43, ls="--", c="gray"); ax[1].text(43.2, 0.05, "dark market\nshutdown", color="gray", fontsize=8)
ax[1].set_xlabel("Time step"); ax[1].set_ylabel("Illicit F1"); ax[1].set_ylim(0, 1); ax[1].set_title("Robustness over time")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/summary.png", dpi=160); plt.show()

counts = dict(n_nodes=int(N), n_edges=int(data.edge_index.shape[1]), n_train=int(tr_m.sum()), n_val=int(va_m.sum()),
              n_test=int(te_m.sum()), illicit_rate_train=float(ytr.mean()), illicit_rate_test=float(yt.mean()))
p, thr = PREDS[best_name]
pre43 = tt < 43
counts["best_model"] = best_name
counts["best_f1_steps_35_42"] = float(f1_score(yt[pre43], (p[pre43] >= thr).astype(int)))
counts["best_f1_steps_43_49"] = float(f1_score(yt[~pre43], (p[~pre43] >= thr).astype(int)))
json.dump(dict(data=counts, models=RES), open("results/results.json", "w"), indent=2, default=float)
counts

In [ ]:
import shutil; shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files; files.download("results.zip")
except ImportError:
    print("saved results.zip")